In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time

load_dotenv()

api_key = os.getenv("RIOT_API_KEY")
print(api_key)

# Get PUUID from gameName and tagLine
def get_puuid(gameName=None, tagLine=None, region=None, api_key=None):
    if gameName is None or tagLine is None:
        raise ValueError("gameName and tagLine must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()['puuid']
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get gameName and tagLine from PUUID
def get_name_and_tag(puuid=None, region=None, api_key=None):
    if puuid is None:
        raise ValueError("puuid must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}riot/account/v1/accounts/by-puuid/{puuid}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data['gameName'], data['tagLine']
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None, None

# Get ladder data for a specific region and league
def get_ladder(region=None, league=None, api_key=None):
    if region is None:
        raise ValueError("region must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/league/v4/{league}leagues/by-queue/RANKED_SOLO_5x5?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match history for a specific PUUID
def get_match_history(puuid=None, region=None, count = None, api_key=None):
    if puuid is None:
        raise ValueError("puuid must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/by-puuid/{puuid}/ids?count={count}&api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match data for a specific match ID
def get_match_data_from_id(match_id=None, region=None, api_key=None):
    if match_id is None:
        raise ValueError("match_id must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/{match_id}?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get match timeline for a specific match ID
def get_match_timeline(match_id=None, region=None, api_key=None):
    if match_id is None:
        raise ValueError("match_id must be provided")
    
    root_url = f"https://{region}.api.riotgames.com/"
    url = f"{root_url}lol/match/v5/matches/{match_id}/timeline?api_key={api_key}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Process match data into a DataFrame
def process_match_to_df(match_data):
    if not match_data:
        return None

    # Metadata & Info
    metadata = match_data.get('metadata', {})
    info = match_data.get('info', {})
    
    match_id = metadata.get('matchId')
    all_participants_puuids = metadata.get('participants', [])
    
    # Stats
    base_info = {
        'matchId': match_id,
        'all_participants': str(all_participants_puuids), 
        'gameCreation': info.get('gameCreation'),
        'gameDuration': info.get('gameDuration'),
        'gameStartTimeStamp': info.get('gameStartTimestamp'),
        'gameEndTimeStamp': info.get('gameEndTimestamp'),
        'PATCH': info.get('gameVersion') # Current patch
    }

    # Bans és Objectives
    teams_stats = {}
    for team in info.get('teams', []):
        t_id = team.get('teamId')
        
        # Bans
        team_bans = [ban.get('championId') for ban in team.get('bans', [])]
        
        # Objectives
        objs = team.get('objectives', {})
        obj_data = {
            'team_baron_kills': objs.get('baron', {}).get('kills'),
            'team_dragon_kills': objs.get('dragon', {}).get('kills'),
            'team_horde_kills': objs.get('horde', {}).get('kills'),
            'team_inhibitor_kills': objs.get('inhibitor', {}).get('kills'),
            'team_riftHerald_kills': objs.get('riftHerald', {}).get('kills'),
            'team_tower_kills': objs.get('tower', {}).get('kills')
        }
        
        teams_stats[t_id] = {
            'bans': team_bans,
            **obj_data
        }

    # ParticipantDto
    rows = []
    for p in info.get('participants', []):
        p_stats = {
            'assists': p.get('assists'),
            'champLevel': p.get('champLevel'),
            'championId': p.get('championId'),
            'deaths': p.get('deaths'),
            'detectorWardsPlaced': p.get('detectorWardsPlaced'),
            'damageDealtToObjectives': p.get('damageDealtToObjectives'),
            'damageSelfMitigated': p.get('damageSelfMitigated'),
            'firstBloodKill': p.get('firstBloodKill'),
            'firstTowerKill': p.get('firstTowerKill'),
            'gameEndedInEarlySurrender': p.get('gameEndedInEarlySurrender'),
            'gameEndedInSurrender': p.get('gameEndedInSurrender'),
            'goldEarned': p.get('goldEarned'),
            'item0': p.get('item0'),
            'item1': p.get('item1'),
            'item2': p.get('item2'),
            'item3': p.get('item3'),
            'item4': p.get('item4'),
            'item5': p.get('item5'),
            'item6': p.get('item6'),
            'kills': p.get('kills'),
            'lane': p.get('lane'),
            'neutralMinionsKilled': p.get('neutralMinionsKilled'),
            'objectivesStolen': p.get('objectivesStolen'),
            'participantId': p.get('participantId'),
            'totalDamageDealtToChampions': p.get('totalDamageDealtToChampions'),
            'totalDamageTaken': p.get('totalDamageTaken'),
            'totalDamageShieldedOnTeammates': p.get('totalDamageShieldedOnTeammates'),
            'visionScore': p.get('visionScore'),
            'win': p.get('win'),
            'teamId': p.get('teamId'),
            'riotIdGameName': p.get('riotIdGameName')
        }
        
        combined_row = {**base_info, **p_stats, **teams_stats.get(p.get('teamId'), {})}
        rows.append(combined_row)

    return pd.DataFrame(rows)

# Get player-specific stats from match data
def get_player_stats_from_match(match_data, target_puuid):
    if not match_data:
        return None

    # Metadata & Info
    metadata = match_data.get('metadata', {})
    info = match_data.get('info', {})
    
    p = next((player for player in info.get('participants', []) if player.get('puuid') == target_puuid), None)
    
    if not p:
        print(f"A játékos ({target_puuid}) nem szerepelt ebben a meccsben.")
        return None

    player_stats = {
        # Match - MetadataDto
        'matchId': metadata.get('matchId'),
        'participants': metadata.get('participants'),
        
        # Match - InfoDto
        'gameCreation': info.get('gameCreation'),
        'gameDuration': info.get('gameDuration'),
        'gameStartTimeStamp': info.get('gameStartTimestamp'),
        'gameEndTimeStamp': info.get('gameEndTimestamp'),
        'PATCH': info.get('gameVersion'),
        
        # ParticipantDto
        'assists': p.get('assists'),
        'champLevel': p.get('champLevel'),
        'championId': p.get('championId'),
        'deaths': p.get('deaths'),
        'detectorWardsPlaced': p.get('detectorWardsPlaced'),
        'damageDealtToObjectives': p.get('damageDealtToObjectives'),
        'damageSelfMitigated': p.get('damageSelfMitigated'),
        'firstBloodKill': p.get('firstBloodKill'),
        'firstTowerKill': p.get('firstTowerKill'),
        'gameEndedInEarlySurrender': p.get('gameEndedInEarlySurrender'),
        'gameEndedInSurrender': p.get('gameEndedInSurrender'),
        'goldEarned': p.get('goldEarned'),
        'item0': p.get('item0'), 'item1': p.get('item1'), 'item2': p.get('item2'),
        'item3': p.get('item3'), 'item4': p.get('item4'), 'item5': p.get('item5'),
        'item6': p.get('item6'),
        'kills': p.get('kills'),
        'lane': p.get('lane'),
        'neutralMinionsKilled': p.get('neutralMinionsKilled'),
        'objectivesStolen': p.get('objectivesStolen'),
        'participantId': p.get('participantId'),
        'totalDamageDealtToChampions': p.get('totalDamageDealtToChampions'),
        'totalDamageTaken': p.get('totalDamageTaken'),
        'totalDamageShieldedOnTeammates': p.get('totalDamageShieldedOnTeammates'),
        'visionScore': p.get('visionScore'),
        'win': p.get('win'),
        'riotIdGameName': p.get('riotIdGameName')
    }

    # Bans & Objectives
    player_team_id = p.get('teamId')
    team_data = next((t for t in info.get('teams', []) if t.get('teamId') == player_team_id), {})
    
    # BanDto
    player_stats['bans'] = [ban.get('championId') for ban in team_data.get('bans', [])]
    
    # ObjectivesDto
    objs = team_data.get('objectives', {})
    player_stats['team_baron_kills'] = objs.get('baron', {}).get('kills')
    player_stats['team_dragon_kills'] = objs.get('dragon', {}).get('kills')
    player_stats['team_horde_kills'] = objs.get('horde', {}).get('kills')
    player_stats['team_inhibitor_kills'] = objs.get('inhibitor', {}).get('kills')
    player_stats['team_riftHerald_kills'] = objs.get('riftHerald', {}).get('kills')
    player_stats['team_tower_kills'] = objs.get('tower', {}).get('kills')

    return player_stats

# Process timeline data into a DataFrame
def process_timeline_to_frames(timeline_data):
    match_id = timeline_data['metadata']['matchId']
    rows = []
    
    # Frame-ek feldolgozása
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        timestamp = frame['timestamp']
        
        # Current data
        for p_id, p_info in frame['participantFrames'].items():
            row = {
                'matchId': match_id,
                'minute': frame_idx,
                'timestamp': timestamp,
                'participantId': int(p_id),
                'totalGold': p_info.get('totalGold'),
                'currentGold': p_info.get('currentGold'),
                'xp': p_info.get('xp'),
                'level': p_info.get('level'),
                'minionsKilled': p_info.get('minionsKilled'),
                'jungleMinionsKilled': p_info.get('jungleMinionsKilled'),
                'positionX': p_info.get('position', {}).get('x'),
                'positionY': p_info.get('position', {}).get('y')
            }
            rows.append(row)
            
    return pd.DataFrame(rows)

# Process timeline events into a DataFrame
def process_timeline_events(timeline_data):
    match_id = timeline_data['metadata']['matchId']
    events_list = []
    
    for frame in timeline_data['info']['frames']:
        for event in frame.get('events', []):
            if event['type'] in ['CHAMPION_KILL', 'ELITE_ monster_kill', 'BUILDING_KILL']: # Csak a fontos eventek
                event_row = {
                    'matchId': match_id,
                    'timestamp': event.get('timestamp'),
                    'type': event.get('type'),
                    'killerId': event.get('killerId'),
                    'victimId': event.get('victimId'),
                    'monsterType': event.get('monsterType'),
                    'teamId': event.get('teamId'),
                    'positionX': event.get('position', {}).get('x'),
                    'positionY': event.get('position', {}).get('y')
                }
                events_list.append(event_row)
                
    return pd.DataFrame(events_list)

RGAPI-984c3f33-2312-4a1a-8cda-0b21f223eb33


In [2]:
player = get_puuid(gameName="Peyz", tagLine="KR11", region="asia", api_key=api_key)
player

'SlQZ2zlvU0VGYNvISzXkK8v6bTF_gFr44fnHfMPZH8QaLZYmMo_XjpJ28nMAlKVBWjtIt3xZUSCA-Q'

In [ ]:
# Regions & Leagues
#regions = ["kr", "euw1", "na1", "br1", "eun1", "jp1", "la1", "la2", "me1", "oc1", "ru", "sg2", "tr1", "tw2", "vn2"]
regions = ["na1"]
leagues = ["challenger", "grandmaster"]

all_ladders = []

# Limit counter
request_count = 0

for region in regions:
    for league in leagues:
        
        # RATE LIMIT
        request_count += 1
        
        # If we hit 100 requests, wait for 125 seconds to respect API limits
        if request_count % 100 == 0:
            print(f"\nLimit reached: {request_count} requests. Waiting for 125 seconds to respect API limits...")
            time.sleep(125)
        # If we hit 20 requests, wait for 2 seconds to avoid hitting short-term limits
        elif request_count % 20 == 0:
            print(f"\n[-] Limit reached: {request_count} requests. Waiting for 2 seconds...")
            time.sleep(2)
        else:
            time.sleep(0.05)

        print(f"Query: {region.upper()} - {league} (Request #{request_count})")
        data = get_ladder(region=region, league=league, api_key=api_key)
        
        if data and 'entries' in data:
            df = pd.DataFrame(data['entries'])
            
            # New columns for region and league
            df['region'] = region
            df['league'] = league
            
            # Append to the list of DataFrames
            all_ladders.append(df)

# Combine all ladders into a single DataFrame
if all_ladders:
    na_challenger_grandmaster_df = pd.concat(all_ladders, ignore_index=True)
    print(f"\nSuccess! {len(na_challenger_grandmaster_df)} players' data loaded.")
else:
    print("\nFailed to retrieve data.")

na_challenger_grandmaster_df.head()

In [ ]:
na_challenger_grandmaster_df.to_csv("na_challenger_grandmaster.csv", index=False)

In [ ]:
try:
    # Read the player data from the CSV file
    players_df = pd.read_csv("na_challenger_grandmaster.csv")
    
except FileNotFoundError:
    print("Error: na_challenger_grandmaster.csv not found.")
    exit()

unique_match_ids = set() # No duplicates
request_count = 0

total_players = len(players_df)
print(f"Total players to process: {total_players}.")

# Iterate through each player and get their match history
for index, row in enumerate(players_df.itertuples(), 1):
    puuid = row.puuid
    # Routing region is based on the server region (na1, kr, euw1, etc.)
    server_region = row.region 
    
    # Region mapping for Match-V5 API 
    if server_region in ['kr', 'jp1', 'tw2', 'vn2']:
        routing_region = "asia"
    elif server_region in ['euw1', 'eun1', 'tr1', 'ru', 'me1']:
        routing_region = "europe"
    elif server_region in ['na1', 'br1', 'la1', 'la2']:
        routing_region = "americas"
    else:
        routing_region = "sea"

    print(f"[{index}/{total_players}] Querying {puuid[:8]}... ({routing_region})")
    
    # Rate Limit Handling
    request_count += 1
    if request_count % 100 == 0:
        print(f"\n[!] 100 requests reached. Waiting for 125 seconds to respect API limits")
        time.sleep(125)
    elif request_count % 20 == 0:
        time.sleep(2)
    else:
        time.sleep(0.05)
    
    # Maximum 100 matches per request
    matches = get_match_history(puuid=puuid, region=routing_region, count=100, api_key=api_key)
    
    if matches:
        # Update the set of unique match IDs with the new matches
        unique_match_ids.update(matches)
    elif matches is None:
         # Request failed (Riot Server error)
         time.sleep(5)

matches_df = pd.DataFrame(list(unique_match_ids), columns=['matchId'])

# Save the unique match IDs to a CSV file
matches_df.to_csv("na_matches.csv", index=False)
print("The unique match IDs have been saved to 'na_matches.csv'.")

matches_df.head()

# Ötletek

Korai előny predikció (win prediction)

Win probability percről percre

Gold és XP regression

Térbeli elemzés (koordinátákkal)

Gold management

Tárgyak győzelmi aránya

# Kérdések

Error 401

Megfelelő adatstruktúra? Formátum stb

Hol tároljam, lesz gép?

Kb mennyi adat legyen?

Térbeli elemzés vészes, tartsuk meg későbbre

In [ ]:
# 1. Töltsük be a meccs ID-kat
matches_df = pd.read_csv("kr_matches.csv")
match_ids = matches_df['matchId'].tolist()

# Teszteléshez érdemes először csak 50-100 meccset nézni:
# match_ids = match_ids[:100] 

print(f"Összesen {len(match_ids)} meccs feldolgozása indul...")

# Számlálók és tárolók
request_count = 0
batch_size = 100 # Hány meccsenként írjuk ki a fájlba?
all_frames = []
all_events = []
batch_index = 1

# 2. Végigiterálunk a meccseken
for index, match_id in enumerate(match_ids, 1):
    print(f"[{index}/{len(match_ids)}] Timeline lekérése: {match_id}...")
    
    # --- Rate Limit Kezelés ---
    request_count += 1
    if request_count % 100 == 0:
        print("\n[!] 100 kérés elérve. 125 mp pihenő az API limit miatt...")
        time.sleep(125)
    elif request_count % 20 == 0:
        time.sleep(2)
    else:
        time.sleep(0.05)
    # --------------------------
    
    # Korea meccsek routingja a Match-V5-nél mindig 'asia'
    timeline_data = get_match_timeline(match_id=match_id, region="asia", api_key=api_key)
    
    if timeline_data and 'info' in timeline_data:
        try:
            frames_df = process_timeline_to_frames(timeline_data)
            events_df = process_timeline_events(timeline_data)
            
            all_frames.append(frames_df)
            all_events.append(events_df)
        except Exception as e:
            print(f"Hiba a feldolgozás során ({match_id}): {e}")
    else:
        print(f"Nem sikerült letölteni vagy üres a válasz: {match_id}")

    # --- CHUNKING (Szakaszos mentés memóriavédelem miatt) ---
    # Ha elértük a 100 meccset (vagy a lista legvégét)
    if index % batch_size == 0 or index == len(match_ids):
        print(f"\n--- Mentés folyamatban (Batch {batch_index}) ---")
        
        if all_frames:
            batch_frames_df = pd.concat(all_frames, ignore_index=True)
            batch_events_df = pd.concat(all_events, ignore_index=True)
            
            # Fájlba írás - "Append" (hozzáfűző) módban!
            # Így nem írja felül a korábbiakat, hanem egy nagy fájlt építünk
            write_header = (batch_index == 1) # Csak a legelső batch-nél írunk fejlécet az oszlopnevekkel
            
            batch_frames_df.to_csv("timeline_frames_kr.csv", mode='a', index=False, header=write_header)
            batch_events_df.to_csv("timeline_events_kr.csv", mode='a', index=False, header=write_header)
            
            print(f"Batch {batch_index} mentve! (+{len(batch_frames_df)} frame sor, +{len(batch_events_df)} event sor)")
            
            # KIÜRÍTJÜK A MEMÓRIÁT a következő 100 meccshez
            all_frames = []
            all_events = []
            batch_index += 1
            
print("\n--- MINDEN MECCS FELDOLGOZVA ÉS ELMENTVE! ---")

In [4]:
def process_timeline_to_frames_full(timeline_data):
    if not timeline_data or 'info' not in timeline_data:
        return None
        
    match_id = timeline_data['metadata']['matchId']
    rows = []
    
    # Frame-ek feldolgozása
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        timestamp = frame['timestamp']
        
        # Current data for each participant
        for p_id, p_info in frame['participantFrames'].items():
            
            # 1. Alapadatok
            row = {
                'matchId': match_id,
                'minute': frame_idx,
                'timestamp': timestamp,
                'participantId': int(p_id),
                'totalGold': p_info.get('totalGold'),
                'currentGold': p_info.get('currentGold'),
                'xp': p_info.get('xp'),
                'level': p_info.get('level'),
                'minionsKilled': p_info.get('minionsKilled'),
                'jungleMinionsKilled': p_info.get('jungleMinionsKilled'),
                'timeEnemySpentControlled': p_info.get('timeEnemySpentControlled')
            }
            
            # 2. Pozíció (X, Y)
            pos = p_info.get('position', {})
            row['positionX'] = pos.get('x')
            row['positionY'] = pos.get('y')
            
            # 3. Champion Stats (Kilapítva)
            # Ezek a hős pillanatnyi statisztikái (pl. AP, AD, Armor) az adott percben
            champ_stats = p_info.get('championStats', {})
            for stat_name, stat_value in champ_stats.items():
                row[f'stat_{stat_name}'] = stat_value
                
            # 4. Damage Stats (Kilapítva)
            # Ezek a halmozott sebzés adatok (fizikai, varázs, true damage)
            damage_stats = p_info.get('damageStats', {})
            for dmg_name, dmg_value in damage_stats.items():
                row[f'dmg_{dmg_name}'] = dmg_value
                
            rows.append(row)
            
    return pd.DataFrame(rows)

In [6]:
def process_timeline_events_full(timeline_data):
    if not timeline_data or 'info' not in timeline_data:
        return None

    match_id = timeline_data['metadata']['matchId']
    events_list = []
    
    # Végigmegyünk a frame-eken (perceken)
    for frame_idx, frame in enumerate(timeline_data['info']['frames']):
        
        # Végigmegyünk az adott percben történt összes eseményen
        for event in frame.get('events', []):
            
            # Alapadatok, amik minden eventhez kellenek
            event_row = {
                'matchId': match_id,
                'minute': frame_idx, # Hanyadik percben (frame) történt
            }
            
            # Dinamikus adatkinyerés: Végigmegyünk az esemény összes mezőjén!
            for key, value in event.items():
                
                # 1. Pozíció szétbontása (x, y)
                if key == 'position' and isinstance(value, dict):
                    event_row['positionX'] = value.get('x')
                    event_row['positionY'] = value.get('y')
                    
                # 2. Listák kezelése (pl. assistingParticipantIds: [2, 3, 5])
                # A CSV nem szereti a listákat, ezért stringgé (szöveggé) alakítjuk
                elif isinstance(value, list):
                    event_row[key] = str(value)
                    
                # 3. Egyéb beágyazott szótárak (ritka az eventeknél, de biztosra megyünk)
                elif isinstance(value, dict):
                    for sub_key, sub_val in value.items():
                        event_row[f"{key}_{sub_key}"] = sub_val
                        
                # 4. Minden más (számok, szövegek, boolean értékek)
                else:
                    event_row[key] = value
                    
            events_list.append(event_row)
            
    return pd.DataFrame(events_list)

In [ ]:
# Fájlnevek beállítása
INPUT_CSV = "kr_matches_40k.csv"
OUTPUT_FRAMES_CSV = "timeline_frames_full_kr.csv"
OUTPUT_EVENTS_CSV = "timeline_events_full_kr.csv"

# 1. Beolvassuk a meccs ID-kat a CSV-ből
try:
    matches_df = pd.read_csv(INPUT_CSV)
    col_name = 'matchId' if 'matchId' in matches_df.columns else matches_df.columns[0]
    match_ids = matches_df[col_name].dropna().unique().tolist()
    # Folytatás
    match_ids = match_ids[3100:]
    print(f"Sikeresen beolvasva {len(match_ids)} db egyedi meccs ID a '{INPUT_CSV}' fájlból.")
except FileNotFoundError:
    print(f"Hiba: A '{INPUT_CSV}' fájl nem található! Ellenőrizd a nevet.")
    match_ids = []

if match_ids:
    # Változók a Rate Limithez és a Batch feldolgozáshoz
    request_count = 0
    batch_size = 50 # 50 meccsenként mentünk és ürítjük a memóriát
    all_frames = []
    all_events = []
    batch_index = 1

    print("\nAdatgyűjtés és feldolgozás indítása...")

    # 2. Végigiterálunk az összes meccsen
    for index, match_id in enumerate(match_ids, 1):
        print(f"[{index}/{len(match_ids)}] Timeline lekérése: {match_id}...")
        
        # --- RATE LIMIT KEZELÉS ---
        request_count += 1
        if request_count % 100 == 0:
            print("\n[!] 100 kérés elérve. 125 mp pihenő az API limit miatt...")
            time.sleep(125)
        elif request_count % 20 == 0:
            time.sleep(2)
        else:
            time.sleep(0.05) # Pici szünet a spammelés elkerülésére
        # --------------------------
        
        # 3. API hívás (Korea mindig 'asia' routing region)
        timeline_data = get_match_timeline(match_id=match_id, region="asia", api_key=api_key)
        
        if timeline_data and 'info' in timeline_data:
            try:
                # Feldolgozás a FULL függvénnyel
                frames_df = process_timeline_to_frames_full(timeline_data)
                events_df = process_timeline_events_full(timeline_data)
                
                if frames_df is not None and not frames_df.empty:
                    all_frames.append(frames_df)
                if events_df is not None and not events_df.empty:
                    all_events.append(events_df)
                    
            except Exception as e:
                print(f"Hiba a {match_id} feldolgozása közben: {e}")
        else:
            print(f"Üres válasz vagy letöltési hiba: {match_id}")

        # 4. BATCH MENTÉS (Szakaszos kiírás fájlba)
        # Ha elértük az 50 meccset, VAGY a lista legvégére értünk
        if index % batch_size == 0 or index == len(match_ids):
            print(f"\n--- Batch {batch_index} mentése folyamatban... ---")
            
            # Fejlécet (header) csak akkor írunk, ha még nem létezik a fájl, 
            # vagy ez a legelső batch a futtatás során
            write_header = not os.path.exists(OUTPUT_FRAMES_CSV)
            
            if all_frames:
                batch_frames_df = pd.concat(all_frames, ignore_index=True)
                # 'mode=a' jelenti az Append-et (hozzáfűzést)
                batch_frames_df.to_csv(OUTPUT_FRAMES_CSV, mode='a', index=False, header=write_header)
                frames_len = len(batch_frames_df)
            else:
                frames_len = 0
                
            if all_events:
                batch_events_df = pd.concat(all_events, ignore_index=True)
                batch_events_df.to_csv(OUTPUT_EVENTS_CSV, mode='a', index=False, header=not os.path.exists(OUTPUT_EVENTS_CSV))
                events_len = len(batch_events_df)
            else:
                events_len = 0
                
            print(f"Batch {batch_index} mentve! (+{frames_len} perces adat, +{events_len} esemény)")
            
            # --- MEMÓRIA ÜRÍTÉSE ---
            all_frames = []
            all_events = []
            batch_index += 1
            
    print("Kész")

Sikeresen beolvasva 38520 db egyedi meccs ID a 'kr_matches_40k.csv' fájlból.

Adatgyűjtés és feldolgozás indítása...
[1/38520] Timeline lekérése: KR_8114799736...
[2/38520] Timeline lekérése: KR_8082690113...
[3/38520] Timeline lekérése: KR_8119442210...
[4/38520] Timeline lekérése: KR_8123819473...
[5/38520] Timeline lekérése: KR_8127027339...
[6/38520] Timeline lekérése: KR_8109667069...
[7/38520] Timeline lekérése: KR_8099713988...
[8/38520] Timeline lekérése: KR_8122416870...
[9/38520] Timeline lekérése: KR_8081964784...
[10/38520] Timeline lekérése: KR_8050874806...
[11/38520] Timeline lekérése: KR_8127615114...
[12/38520] Timeline lekérése: KR_8059043408...
[13/38520] Timeline lekérése: KR_8106696049...
[14/38520] Timeline lekérése: KR_8114069626...
[15/38520] Timeline lekérése: KR_8125659904...
[16/38520] Timeline lekérése: KR_8120808618...
[17/38520] Timeline lekérése: KR_8113116299...
[18/38520] Timeline lekérése: KR_8087095707...
[19/38520] Timeline lekérése: KR_8091121635...